# Laboratório III · Veículos: ETFs, futuros e execução
*Bitcoin na Veia: do halving ao ETF*, 2ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. O laboratório compara os ETFs dos EUA e da B3 com a moeda, mede o custo da rolagem num ETF de futuros, o peso da taxa de administração com os anos, a cesta contra o Bitcoin e a Strategy contra o Bitcoin, calcula o imposto de cada caminho e, para os derivativos, mede a taxa de financiamento, a chance de liquidação, o ajuste diário e o custo de uma put; para a execução, lê o livro de ofertas, testa o stop no pior minuto de 10/10/2025 e o stop móvel desde o halving. Serve aos Capítulos 9, 10 e 11. O Capítulo 19 da *Análise Técnica dos Mercados Financeiros*, 4ª edição, usa a mesma base de dados.

In [ ]:
!pip install -q yfinance

In [2]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
RAW = "https://raw.githubusercontent.com/traderbrasil/livro-bitcoin-na-veia/main/labs/dados/"
def baixa(nome):
    """Usa a cópia dos dados salva no GitHub do livro (mesmos números do livro); sem ela, baixa tudo de novo."""
    if not os.path.exists(nome):
        try:
            import urllib.request
            urllib.request.urlretrieve(RAW + nome, nome)
        except Exception:
            pass
    return os.path.exists(nome)
CACHE = "dados_lab3.pkl"
pc = lambda x, k=0: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=0: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=0: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
TICKERS = ["BTC-USD", "ETH-USD", "BRL=X", "IBIT", "FBTC", "GBTC", "BTC", "ETHA", "BITO", "MSTR", "QBTC11.SA",
           "BITH11.SA", "HASH11.SA", "HODL11.SA", "ETHE11.SA", "QETH11.SA", "BTC=F", "^IXIC"]

if baixa(CACHE):
    D = pd.read_pickle(CACHE)
else:
    d = yf.download(TICKERS, start="2014-01-01", end=FIM, auto_adjust=True, progress=False)
    D = {k: d[k] for k in ("Open", "High", "Low", "Close", "Volume")}
    pd.to_pickle(D, CACHE)
C = D["Close"]
btc = C["BTC-USD"].dropna(); eth = C["ETH-USD"].dropna(); BRL = C["BRL=X"].dropna()
BTC_BRL = (btc * BRL.reindex(btc.index).ffill()).dropna()
ETH_BRL = (eth * BRL.reindex(eth.index).ffill()).dropna()
print(f"Bitcoin em 30/09/2026: US$ {nr(btc.iloc[-1])} | R$ {nr(BTC_BRL.iloc[-1])}")

Bitcoin em 30/09/2026: US$ 83.554 | R$ 434.722


## Parte 1: o ETF acompanha a moeda?

In [3]:
def compara(prod, ref, ini=None):
    dd = pd.DataFrame({"e": C[prod], "b": ref}).dropna()
    if ini: dd = dd.loc[ini:]
    r = dd.pct_change().dropna()
    te, tb = dd["e"].iloc[-1] / dd["e"].iloc[0] - 1, dd["b"].iloc[-1] / dd["b"].iloc[0] - 1
    anos = (dd.index[-1] - dd.index[0]).days / 365.25
    ae, ab = (1 + te) ** (1 / anos) - 1, (1 + tb) ** (1 / anos) - 1
    return dd.index[0], r["e"].corr(r["b"]), te, tb, ae - ab
TAB_ETF = []
print("Parte 1. Produto contra a moeda (retornos nos dias em que os dois têm preço; até 30/09/2026)")
for prod, ref, nome, ini in [("IBIT", btc, "IBIT (EUA) x Bitcoin em dólar", None), ("FBTC", btc, "FBTC (EUA) x Bitcoin em dólar", None),
                             ("GBTC", btc, "GBTC (EUA, desde a conversão) x Bitcoin em dólar", "2024-01-11"),
                             ("ETHA", eth, "ETHA (EUA) x Ether em dólar", None),
                             ("QBTC11.SA", BTC_BRL, "QBTC11 (B3) x Bitcoin em reais", None),
                             ("BITH11.SA", BTC_BRL, "BITH11 (B3) x Bitcoin em reais", None),
                             ("ETHE11.SA", ETH_BRL, "ETHE11 (B3) x Ether em reais", None)]:
    i0, cor, te, tb, dif = compara(prod, ref, ini)
    TAB_ETF.append((nome, i0, cor, te, tb, dif))
    print(f"   {nome:<48} desde {i0:%d/%m/%Y}: correlação {nr(cor, 2)} | produto {ps(te)} x moeda {ps(tb)}"
          f" | diferença por ano {ps(dif, 2)}")
print("   A diferença vem da taxa de administração, do câmbio e do horário: a moeda fecha à meia-noite (UTC), o ETF no fim do pregão.")

Parte 1. Produto contra a moeda (retornos nos dias em que os dois têm preço; até 30/09/2026)
   IBIT (EUA) x Bitcoin em dólar                    desde 11/01/2024: correlação 0,91 | produto +78% x moeda +80% | diferença por ano -0,62%
   FBTC (EUA) x Bitcoin em dólar                    desde 11/01/2024: correlação 0,91 | produto +78% x moeda +80% | diferença por ano -0,64%
   GBTC (EUA, desde a conversão) x Bitcoin em dólar desde 11/01/2024: correlação 0,91 | produto +76% x moeda +80% | diferença por ano -1,14%
   ETHA (EUA) x Ether em dólar                      desde 23/07/2024: correlação 0,91 | produto -23% x moeda -23% | diferença por ano -0,21%
   QBTC11 (B3) x Bitcoin em reais                   desde 23/06/2021: correlação 0,80 | produto +145% x moeda +160% | diferença por ano -1,37%
   BITH11 (B3) x Bitcoin em reais                   desde 04/08/2021: correlação 0,81 | produto +97% x moeda +110% | diferença por ano -1,49%
   ETHE11 (B3) x Ether em reais                     desde 

## Parte 2: ETF de futuros e o custo da rolagem

In [4]:
i0, cor, te, tb, dif = compara("BITO", btc)
print(f"Parte 2. BITO (futuros da CME) x Bitcoin em dólar, desde {i0:%d/%m/%Y}: produto {ps(te)} x moeda {ps(tb)}"
      f" | diferença por ano {ps(dif, 1)} | correlação {nr(cor, 2)}")
i0, cor, te, tb, dif = compara("BTC=F", btc)
print(f"   Futuro contínuo da CME x Bitcoin, desde {i0:%d/%m/%Y}: {ps(te)} x {ps(tb)} (a série contínua emenda os vencimentos e esconde a rolagem)")
print("   O ETF de futuros vende o contrato que vence e compra o seguinte, mais caro quando o mercado está em contango. Essa troca, todo mês, é o custo.")

Parte 2. BITO (futuros da CME) x Bitcoin em dólar, desde 20/10/2021: produto -3% x moeda +27% | diferença por ano -5,6% | correlação 0,91
   Futuro contínuo da CME x Bitcoin, desde 18/12/2017: +339% x +337% (a série contínua emenda os vencimentos e esconde a rolagem)
   O ETF de futuros vende o contrato que vence e compra o seguinte, mais caro quando o mercado está em contango. Essa troca, todo mês, é o custo.


## Parte 3: a taxa de administração com os anos

In [5]:
TAXAS = [("Mini Trust da Grayscale (BTC)", 0.0015), ("IBIT, FBTC, ETHA", 0.0025), ("HODL11 (total)", 0.0040),
         ("QBTC11, BITH11, ETHE11", 0.0070), ("HASH11 (total, com o fundo investido)", 0.0130), ("GBTC", 0.0150)]
print("Parte 3. Quanto da moeda a taxa come (sem contar câmbio, horário ou spread)")
for nome, t in TAXAS:
    print(f"   {nome:<42} {pc(t, 2):>6} a.a. -> 5 anos: {pc(1 - (1 - t) ** 5, 1)} | 10 anos: {pc(1 - (1 - t) ** 10, 1)} | 20 anos: {pc(1 - (1 - t) ** 20, 1)}")
print("   Taxas de out/2026, conferidas nos sites das gestoras. Confira antes de comprar: elas mudam.")

Parte 3. Quanto da moeda a taxa come (sem contar câmbio, horário ou spread)
   Mini Trust da Grayscale (BTC)               0,15% a.a. -> 5 anos: 0,7% | 10 anos: 1,5% | 20 anos: 3,0%
   IBIT, FBTC, ETHA                            0,25% a.a. -> 5 anos: 1,2% | 10 anos: 2,5% | 20 anos: 4,9%
   HODL11 (total)                              0,40% a.a. -> 5 anos: 2,0% | 10 anos: 3,9% | 20 anos: 7,7%
   QBTC11, BITH11, ETHE11                      0,70% a.a. -> 5 anos: 3,5% | 10 anos: 6,8% | 20 anos: 13,1%
   HASH11 (total, com o fundo investido)       1,30% a.a. -> 5 anos: 6,3% | 10 anos: 12,3% | 20 anos: 23,0%
   GBTC                                        1,50% a.a. -> 5 anos: 7,3% | 10 anos: 14,0% | 20 anos: 26,1%
   Taxas de out/2026, conferidas nos sites das gestoras. Confira antes de comprar: elas mudam.


## Parte 4: a cesta contra o Bitcoin

In [6]:
i0, cor, te, tb, dif = compara("HASH11.SA", BTC_BRL)
print(f"Parte 4. HASH11 (cesta, Nasdaq Crypto Index) x Bitcoin em reais, desde {i0:%d/%m/%Y}: cesta {ps(te)} x bitcoin {ps(tb)}"
      f" | diferença por ano {ps(dif, 1)} | correlação {nr(cor, 2)}")
print("   Não é erro de rastreamento: a cesta segue o seu índice. A diferença mostra o custo de diversificar em altcoins nesse período (Capítulo 6).")

Parte 4. HASH11 (cesta, Nasdaq Crypto Index) x Bitcoin em reais, desde 26/04/2021: cesta +7% x bitcoin +47% | diferença por ano -6,1% | correlação 0,81
   Não é erro de rastreamento: a cesta segue o seu índice. A diferença mostra o custo de diversificar em altcoins nesse período (Capítulo 6).


## Parte 5: empresa com bitcoin no caixa não é ETF

In [7]:
dd = pd.DataFrame({"m": C["MSTR"], "b": btc}).dropna().loc["2020-08-11":]   # 11/08/2020: primeira compra de bitcoins anunciada
r = dd.pct_change().dropna()
print(f"Parte 5. Strategy (MSTR) x Bitcoin, de {dd.index[0]:%d/%m/%Y} a 30/09/2026")
for c, nome in [("m", "MSTR"), ("b", "Bitcoin")]:
    s = dd[c]; q = s / s.cummax() - 1
    print(f"   {nome:<8} retorno {ps(s.iloc[-1] / s.iloc[0] - 1)} | volatilidade anual {pc(r[c].std() * np.sqrt(252))}"
          f" | maior queda {pc(q.min())} | do topo de {s.idxmax():%d/%m/%Y} até 30/09/2026: {pc(s.iloc[-1] / s.max() - 1)}")
print(f"   beta da MSTR contra o Bitcoin: {nr(r['m'].cov(r['b']) / r['b'].var(), 2)} | correlação {nr(r['m'].corr(r['b']), 2)}")
print("   A ação soma ao Bitcoin o risco da empresa: dívida, emissão de ações e o prêmio ou desconto sobre os bitcoins que ela tem (mNAV).")

Parte 5. Strategy (MSTR) x Bitcoin, de 11/08/2020 a 30/09/2026
   MSTR     retorno +1035% | volatilidade anual 93% | maior queda -89% | do topo de 20/11/2024 até 30/09/2026: -68%
   Bitcoin  retorno +632% | volatilidade anual 58% | maior queda -77% | do topo de 06/10/2025 até 30/09/2026: -33%
   beta da MSTR contra o Bitcoin: 1,14 | correlação 0,71
   A ação soma ao Bitcoin o risco da empresa: dívida, emissão de ações e o prêmio ou desconto sobre os bitcoins que ela tem (mNAV).


## Parte 6: o imposto de cada caminho

In [8]:
INVESTIDO, GANHO_PCT = 100_000, 0.50          # R$ investidos e ganho na venda (troque pelos seus)
IOF_REMESSA = 0.011                             # remessa para investimento no exterior (Decreto 12.499/2025)
ganho = INVESTIDO * GANHO_PCT
venda = INVESTIDO + ganho
print(f"Parte 6. R$ {nr(INVESTIDO)} investidos, venda por R$ {nr(venda)} (ganho de R$ {nr(ganho)})")
print(f"   ETF na B3, operação comum: 15% = R$ {nr(0.15 * ganho)} (DARF 6015, mês seguinte; prejuízo compensa ações e ETFs)")
print(f"   ETF na B3, day trade: 20% = R$ {nr(0.20 * ganho)}")
print(f"   Moeda em corretora nacional, venda única acima de R$ 35 mil no mês: 15% = R$ {nr(0.15 * ganho)} (DARF 1897)")
meses = int(np.ceil(venda / 35_000))
print(f"   Moeda em corretora nacional, vendendo até R$ 35 mil por mês: isento, mas a venda leva {meses} meses, com o preço mudando no caminho")
print(f"   ETF nos EUA (Lei 14.754/2023): 15% na declaração anual = R$ {nr(0.15 * ganho)}, mais IOF de {pc(IOF_REMESSA, 1)} na ida = R$ {nr(IOF_REMESSA * INVESTIDO)}")
print("   No exterior, o ganho é medido em reais: a variação do dólar entra na conta. Detalhes, prazos e exceções no Capítulo 17.")

Parte 6. R$ 100.000 investidos, venda por R$ 150.000 (ganho de R$ 50.000)
   ETF na B3, operação comum: 15% = R$ 7.500 (DARF 6015, mês seguinte; prejuízo compensa ações e ETFs)
   ETF na B3, day trade: 20% = R$ 10.000
   Moeda em corretora nacional, venda única acima de R$ 35 mil no mês: 15% = R$ 7.500 (DARF 1897)
   Moeda em corretora nacional, vendendo até R$ 35 mil por mês: isento, mas a venda leva 5 meses, com o preço mudando no caminho
   ETF nos EUA (Lei 14.754/2023): 15% na declaração anual = R$ 7.500, mais IOF de 1,1% na ida = R$ 1.100
   No exterior, o ganho é medido em reais: a variação do dólar entra na conta. Detalhes, prazos e exceções no Capítulo 17.


## Parte 7: a taxa de financiamento dos perpétuos (Capítulo 10)

In [9]:
import io, zipfile, urllib.request
FCACHE = "funding_btcusdt.pkl"
if baixa(FCACHE):
    F = pd.read_pickle(FCACHE)
else:   # arquivos públicos da Binance (data.binance.vision), contrato perpétuo BTCUSDT, um por mês
    partes = []
    for per in pd.period_range("2020-01", "2026-09", freq="M"):
        u = f"https://data.binance.vision/data/futures/um/monthly/fundingRate/BTCUSDT/BTCUSDT-fundingRate-{per.year}-{per.month:02d}.zip"
        z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(u, timeout=60).read()))
        partes.append(pd.read_csv(z.open(z.namelist()[0])))
    F = pd.concat(partes); F.to_pickle(FCACHE)
fr = F.assign(t=pd.to_datetime(F["calc_time"], unit="ms")).set_index("t").sort_index()["last_funding_rate"]
print("Parte 7. Taxa de financiamento do perpétuo BTCUSDT (Binance), paga a cada 8 horas")
print("   Positiva: o comprado paga ao vendido. A soma do ano é o custo de carregar 1 dólar comprado o ano inteiro.")
for ano, x in fr.groupby(fr.index.year):
    print(f"   {ano}{'*' if ano == 2026 else ' '}: soma {pc(x.sum(), 1):>6} | períodos negativos {pc((x < 0).mean()):>4} | maior {pc(x.max(), 2)} | menor {pc(x.min(), 2)}")
dia = fr.resample("D").sum()
print(f"   * 2026 até 30/09. Dias mais negativos: " + ", ".join(f"{d:%d/%m/%Y} ({pc(v, 2)})" for d, v in dia.nsmallest(2).items()))
print("   O pessimismo extremo pagou quem estava comprado: na Covid (mar/2020) e na quebra da FTX (nov/2022).")

Parte 7. Taxa de financiamento do perpétuo BTCUSDT (Binance), paga a cada 8 horas
   Positiva: o comprado paga ao vendido. A soma do ano é o custo de carregar 1 dólar comprado o ano inteiro.
   2020 : soma  17,2% | períodos negativos  14% | maior 0,30% | menor -0,30%
   2021 : soma  30,6% | períodos negativos   7% | maior 0,25% | menor -0,09%
   2022 : soma   4,2% | períodos negativos  22% | maior 0,01% | menor -0,12%
   2023 : soma   7,9% | períodos negativos  10% | maior 0,06% | menor -0,01%
   2024 : soma  12,0% | períodos negativos   8% | maior 0,09% | menor -0,01%
   2025 : soma   5,1% | períodos negativos  13% | maior 0,01% | menor -0,01%
   2026*: soma   2,2% | períodos negativos  26% | maior 0,01% | menor -0,02%
   * 2026 até 30/09. Dias mais negativos: 13/03/2020 (-0,38%), 10/11/2022 (-0,25%)
   O pessimismo extremo pagou quem estava comprado: na Covid (mar/2020) e na quebra da FTX (nov/2022).


## Parte 8: alavancagem e liquidação (Capítulo 10)

In [10]:
H, MM = 30, 0.005          # janela de 30 dias e margem de manutenção de 0,5% (hipóteses)
c = C["BTC-USD"].dropna().loc["2017-01-01":]
lo = D["Low"]["BTC-USD"].reindex(c.index).values; hi = D["High"]["BTC-USD"].reindex(c.index).values; cv = c.values
TAB_LIQ = []
print(f"Parte 8. Chance de ser liquidado em até {H} dias, abrindo no fechamento de qualquer dia de 2017 a 2026")
for L in [2, 3, 5, 10, 20, 50, 100]:
    n = len(cv) - H
    comp = np.mean([lo[i + 1:i + 1 + H].min() <= cv[i] * (1 - 1 / L + MM) for i in range(n)])
    vend = np.mean([hi[i + 1:i + 1 + H].max() >= cv[i] * (1 + 1 / L - MM) for i in range(n)])
    TAB_LIQ.append((L, comp, vend))
    print(f"   {L:>3}x: comprado {pc(comp, 1):>6} | vendido {pc(vend, 1):>6}")
print("   Usa a máxima e a mínima diárias, sem taxa de financiamento: a conta real é pior.")

Parte 8. Chance de ser liquidado em até 30 dias, abrindo no fechamento de qualquer dia de 2017 a 2026
     2x: comprado   1,2% | vendido   7,6%
     3x: comprado   6,4% | vendido  16,3%
     5x: comprado  22,3% | vendido  32,5%
    10x: comprado  45,7% | vendido  58,6%
    20x: comprado  68,6% | vendido  78,2%


    50x: comprado  86,4% | vendido  90,7%
   100x: comprado  93,9% | vendido  96,1%
   Usa a máxima e a mínima diárias, sem taxa de financiamento: a conta real é pior.


## Parte 9: ajuste diário com perda, na B3 (Capítulo 10)

In [11]:
QTD = 0.1                   # 10 contratos de 0,01 BTC
p = BTC_BRL.loc["2025-10-06":"2025-10-17"]
p = p[p.index.dayofweek < 5]                     # a B3 não abre no fim de semana
print(f"Parte 9. Compra de 10 contratos (0,1 BTC) no fechamento de 06/10/2025, a máxima histórica. Valor: R$ {nr(p.iloc[0] * QTD)}")
acum = 0
for (d0, v0), (d1, v1) in zip(p.iloc[:-1].items(), p.iloc[1:].items()):
    aj = (v1 - v0) * QTD; acum += aj
    print(f"   {d1:%d/%m}: bitcoin R$ {nr(v1)} | ajuste R$ {nr(aj):>7} | acumulado R$ {nr(acum):>7}")
for m in [0.20, 0.10]:
    marg = p.iloc[0] * QTD * m
    print(f"   Com margem de {pc(m)} (R$ {nr(marg)}), a perda consome {pc(-acum / marg)} da margem")
print("   Aproximação com o fechamento diário da moeda; o ajuste real usa o preço de referência da B3. Margens são hipóteses.")

Parte 9. Compra de 10 contratos (0,1 BTC) no fechamento de 06/10/2025, a máxima histórica. Valor: R$ 66.550
   07/10: bitcoin R$ 645.053 | ajuste R$  -2.045 | acumulado R$  -2.045
   08/10: bitcoin R$ 660.417 | ajuste R$   1.536 | acumulado R$    -509
   09/10: bitcoin R$ 649.324 | ajuste R$  -1.109 | acumulado R$  -1.618
   10/10: bitcoin R$ 608.776 | ajuste R$  -4.055 | acumulado R$  -5.673
   13/10: bitcoin R$ 636.642 | ajuste R$   2.787 | acumulado R$  -2.886
   14/10: bitcoin R$ 618.092 | ajuste R$  -1.855 | acumulado R$  -4.741
   15/10: bitcoin R$ 607.258 | ajuste R$  -1.083 | acumulado R$  -5.825
   16/10: bitcoin R$ 589.538 | ajuste R$  -1.772 | acumulado R$  -7.597
   17/10: bitcoin R$ 579.272 | ajuste R$  -1.027 | acumulado R$  -8.623
   Com margem de 20% (R$ 13.310), a perda consome 65% da margem
   Com margem de 10% (R$ 6.655), a perda consome 130% da margem
   Aproximação com o fechamento diário da moeda; o ajuste real usa o preço de referência da B3. Margens são hipótese

## Parte 10: a put como seguro (Capítulo 10)

In [12]:
from math import log, sqrt, exp, erf
N = lambda x: 0.5 * (1 + erf(x / sqrt(2)))
def put_bs(S, K, T, vol, r=0.0):
    d1 = (log(S / K) + (r + vol ** 2 / 2) * T) / (vol * sqrt(T)); d2 = d1 - vol * sqrt(T)
    return K * exp(-r * T) * N(-d2) - S * N(-d1)
S = BTC_BRL.iloc[-1]
VOL = btc.pct_change().loc["2026-07-01":].std() * np.sqrt(365)     # volatilidade realizada de jul a set/2026
K, T = 400_000, 0.25
pr = put_bs(S, K, T, VOL)
print(f"Parte 10. Bitcoin a R$ {nr(S)}; volatilidade realizada {pc(VOL)}; put de strike R$ {nr(K)} por 3 meses")
print(f"   Prêmio teórico (Black-Scholes, sem juros): R$ {nr(pr)} por bitcoin, {pc(pr / S, 1)} do preço")
for queda in [0.0, -0.10, -0.30, -0.50]:
    fim = S * (1 + queda); sem = fim - S; com = fim - S + max(K - fim, 0) - pr
    print(f"   Bitcoin {ps(queda):>5}: sem seguro R$ {nr(sem):>9} | com a put R$ {nr(com):>9}")
print("   O seguro limita a perda a cerca de R$ " + nr(S - K + pr) + " por bitcoin, e custa o prêmio mesmo quando nada acontece.")

Parte 10. Bitcoin a R$ 434.722; volatilidade realizada 39%; put de strike R$ 400.000 por 3 meses
   Prêmio teórico (Black-Scholes, sem juros): R$ 17.875 por bitcoin, 4,1% do preço
   Bitcoin   +0%: sem seguro R$         0 | com a put R$   -17.875
   Bitcoin  -10%: sem seguro R$   -43.472 | com a put R$   -52.597
   Bitcoin  -30%: sem seguro R$  -130.417 | com a put R$   -52.597
   Bitcoin  -50%: sem seguro R$  -217.361 | com a put R$   -52.597
   O seguro limita a perda a cerca de R$ 52.597 por bitcoin, e custa o prêmio mesmo quando nada acontece.


## Parte 11: o livro de ofertas, o spread e o escorregamento (Capítulo 11)

In [13]:
import json, urllib.request
BCACHE = "book_snapshot.pkl"
def pega(u):
    req = urllib.request.Request(u, headers={"User-Agent": "Mozilla/5.0 (Laboratorio Bitcoin na Veia)"})
    return json.loads(urllib.request.urlopen(req, timeout=30).read())
if baixa(BCACHE):       # foto usada no livro; para tirar uma foto nova, troque o nome em BCACHE
    BK = pd.read_pickle(BCACHE)
else:
    BK = {"quando": pd.Timestamp.now(tz="UTC").strftime("%d/%m/%Y %H:%M UTC"),
          "coinbase": pega("https://api.exchange.coinbase.com/products/BTC-USD/book?level=2"),
          "mb": pega("https://api.mercadobitcoin.net/api/v4/BTC-BRL/orderbook?limit=1000")}
    pd.to_pickle(BK, BCACHE)
def livro(lado):
    df = pd.DataFrame([(float(x[0]), float(x[1])) for x in lado], columns=["p", "q"])
    return df
def resumo(bk):
    a = livro(bk["asks"]).sort_values("p"); b = livro(bk["bids"]).sort_values("p", ascending=False)
    meio = (a.p.iloc[0] + b.p.iloc[0]) / 2
    return a, b, meio
def escorrega(a, meio, valor):
    falta, custo, qtd = valor, 0.0, 0.0
    for p, q in zip(a.p, a.q):
        if p * q >= falta:
            qtd += falta / p; custo += falta; falta = 0; break
        qtd += q; custo += p * q; falta -= p * q
    return None if falta > 0 else (custo / qtd) / meio - 1
aC, bC, mC = resumo(BK["coinbase"]); aM, bM, mM = resumo(BK["mb"])
CAMBIO = mM / mC                  # câmbio implícito entre os dois livros no mesmo minuto
print(f"Parte 11. Foto dos livros de ofertas em {BK['quando']}: Coinbase (BTC-USD) e Mercado Bitcoin (BTC-BRL)")
for nome, a, b, m, moeda, fx in [("Coinbase", aC, bC, mC, "US$", 1), ("Mercado Bitcoin", aM, bM, mM, "R$", CAMBIO)]:
    esp = (a.p.iloc[0] - b.p.iloc[0]) / m
    prof_v = (a.p * a.q)[a.p <= m * 1.01].sum() / fx; prof_c = (b.p * b.q)[b.p >= m * 0.99].sum() / fx
    print(f"   {nome:<16} preço médio {moeda} {nr(m)} | spread {moeda} {nr(a.p.iloc[0] - b.p.iloc[0], 2)} ({nr(esp * 1e4, 3)} ponto-base) | até 1% do preço: "
          f"US$ {nr(prof_v / 1e6, 2)} mi à venda e US$ {nr(prof_c / 1e6, 2)} mi à compra")
print(f"   Escorregamento de uma compra a mercado (câmbio implícito de R$ {nr(CAMBIO, 2)} por dólar):")
for usd in [10_000, 100_000, 1_000_000]:
    eC = escorrega(aC, mC, usd); eM = escorrega(aM, mM, usd * CAMBIO)
    print(f"   US$ {nr(usd):>9}: Coinbase {pc(eC, 3):>7} | Mercado Bitcoin " + ("livro insuficiente" if eM is None else pc(eM, 2)))
print("   É uma foto de um minuto. O livro muda o tempo todo e afina em dia de pânico.")

Parte 11. Foto dos livros de ofertas em 26/10/2026 20:25 UTC: Coinbase (BTC-USD) e Mercado Bitcoin (BTC-BRL)
   Coinbase         preço médio US$ 79.807 | spread US$ 0,01 (0,001 ponto-base) | até 1% do preço: US$ 14,76 mi à venda e US$ 15,82 mi à compra


   Mercado Bitcoin  preço médio R$ 431.800 | spread R$ 1,00 (0,023 ponto-base) | até 1% do preço: US$ 0,17 mi à venda e US$ 0,02 mi à compra
   Escorregamento de uma compra a mercado (câmbio implícito de R$ 5,41 por dólar):
   US$    10.000: Coinbase  0,002% | Mercado Bitcoin 0,11%
   US$   100.000: Coinbase  0,007% | Mercado Bitcoin 0,24%
   US$ 1.000.000: Coinbase  0,021% | Mercado Bitcoin 7,82%
   É uma foto de um minuto. O livro muda o tempo todo e afina em dia de pânico.


## Parte 12: o stop no pior minuto, 10/10/2025 (Capítulo 11)

In [14]:
KCACHE = "klines_20251010.pkl"
def velas(sym, dia):
    u = f"https://data.binance.vision/data/spot/daily/klines/{sym}/1m/{sym}-1m-{dia}.zip"
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(u, timeout=60).read()))
    df = pd.read_csv(z.open(z.namelist()[0]), header=None).iloc[:, :6]
    df.columns = ["t", "o", "h", "l", "c", "v"]
    df["t"] = pd.to_datetime(df["t"], unit="us" if df["t"].iloc[0] > 1e14 else "ms")
    return df.set_index("t")
if baixa(KCACHE):
    K = pd.read_pickle(KCACHE)
else:
    K = {s: pd.concat([velas(s, "2025-10-10"), velas(s, "2025-10-11")])
         for s in ["BTCUSDT", "ETHUSDT", "SOLUSDT", "AVAXUSDT", "ATOMUSDT", "SUIUSDT", "DOGEUSDT"]}
    pd.to_pickle(K, KCACHE)
print("Parte 12. Binance à vista, velas de 1 minuto. Referência: abertura das 20:50 UTC de 10/10/2025, o minuto do anúncio da tarifa")
TAB_1010 = []
for s, d in K.items():
    ref = d.loc["2025-10-10 20:50", "o"]; w = d.loc["2025-10-10 20:50":"2025-10-10 23:59"]
    TAB_1010.append((s, ref, w["l"].min(), w["l"].idxmin(), w["c"].iloc[-1]))
    print(f"   {s.replace('USDT', ''):<5} de {nr(ref, 4)} a mínima de {nr(w['l'].min(), 4)} às {w['l'].idxmin():%H:%M} ({ps(w['l'].min() / ref - 1, 1)})"
          f"; fechou o dia em {nr(w['c'].iloc[-1], 4)} ({ps(w['c'].iloc[-1] / ref - 1, 1)})")
print("\n   Stops de venda abaixo da referência. A mercado: pior caso = mínima do minuto do disparo. Limitado: limite 2 pontos abaixo do disparo.")
for s, d in K.items():
    ref = d.loc["2025-10-10 20:50", "o"]; w = d.loc["2025-10-10 20:50":"2025-10-10 23:59"]
    txt = []
    for dist in [0.10, 0.50]:
        st = ref * (1 - dist); bate = w[w["l"] <= st]
        if bate.empty:
            txt.append(f"stop {pc(dist)}: não disparou"); continue
        m = bate.index[0]; lim = st * 0.98
        exec_lim = (w.loc[m:, "h"] >= lim).any()
        txt.append(f"stop {pc(dist)} às {m:%H:%M}: mercado {ps(w.loc[m, 'l'] / ref - 1)}, limitado " + (ps(lim / ref - 1) if exec_lim else "não vendeu"))
    print(f"   {s.replace('USDT', ''):<5} " + " | ".join(txt) + f" | sem stop: {ps(w['c'].iloc[-1] / ref - 1)} no fim do dia")
print("   Num livro vazio, o stop a mercado vende a qualquer preço, até no fundo do pavio; o limitado protege o preço, mas pode não vender se o preço não voltar.")

Parte 12. Binance à vista, velas de 1 minuto. Referência: abertura das 20:50 UTC de 10/10/2025, o minuto do anúncio da tarifa
   BTC   de 116.890,8000 a mínima de 102.000,0000 às 21:20 (-12,7%); fechou o dia em 112.774,5000 (-3,5%)
   ETH   de 4.013,4200 a mínima de 3.435,0000 às 21:20 (-14,4%); fechou o dia em 3.829,7200 (-4,6%)
   SOL   de 207,2300 a mínima de 168,7900 às 21:28 (-18,5%); fechou o dia em 188,4300 (-9,1%)
   AVAX  de 26,5800 a mínima de 8,5200 às 21:21 (-67,9%); fechou o dia em 20,6800 (-22,2%)
   ATOM  de 3,9320 a mínima de 0,0010 às 21:20 (-100,0%); fechou o dia em 2,9460 (-25,1%)
   SUI   de 3,2604 a mínima de 0,5597 às 21:21 (-82,8%); fechou o dia em 2,6047 (-20,1%)
   DOGE  de 0,2339 a mínima de 0,0950 às 21:20 (-59,4%); fechou o dia em 0,1935 (-17,3%)

   Stops de venda abaixo da referência. A mercado: pior caso = mínima do minuto do disparo. Limitado: limite 2 pontos abaixo do disparo.
   BTC   stop 10% às 21:19: mercado -11%, limitado -12% | stop 50%: não dispa

## Parte 13: o stop móvel desde o halving de 2024 (Capítulo 11)

In [15]:
print("Parte 13. Compra no fechamento de 19/04/2024 (véspera do halving) e stop móvel sobre o fechamento diário, até 30/09/2026")
serie = btc.loc["2024-04-19":]
for dist in [0.10, 0.15, 0.20, 0.25, 0.30]:
    topo = serie.iloc[0]; saida = None
    for d, v in serie.items():
        topo = max(topo, v)
        if v <= topo * (1 - dist):
            saida = (d, v, topo); break
    if saida:
        d, v, t = saida
        print(f"   stop de {pc(dist):>3}: saiu em {d:%d/%m/%Y} a US$ {nr(v)} ({ps(v / serie.iloc[0] - 1)} desde a compra; o topo até ali foi US$ {nr(t)})")
    else:
        print(f"   stop de {pc(dist):>3}: não saiu; em 30/09/2026, {ps(serie.iloc[-1] / serie.iloc[0] - 1)}")
print(f"   Sem stop: {ps(serie.iloc[-1] / serie.iloc[0] - 1)}. Stop curto sai cedo nas correções; stop longo devolve mais do topo. Não há distância certa, há a que combina com o seu prazo.")

Parte 13. Compra no fechamento de 19/04/2024 (véspera do halving) e stop móvel sobre o fechamento diário, até 30/09/2026
   stop de 10%: saiu em 01/05/2024 a US$ 58.254 (-9% desde a compra; o topo até ali foi US$ 66.838)
   stop de 15%: saiu em 24/06/2024 a US$ 60.277 (-6% desde a compra; o topo até ali foi US$ 71.448)
   stop de 20%: saiu em 04/07/2024 a US$ 56.978 (-11% desde a compra; o topo até ali foi US$ 71.448)
   stop de 25%: saiu em 10/03/2025 a US$ 78.532 (+23% desde a compra; o topo até ali foi US$ 106.146)
   stop de 30%: saiu em 20/11/2025 a US$ 86.632 (+36% desde a compra; o topo até ali foi US$ 124.753)
   Sem stop: +31%. Stop curto sai cedo nas correções; stop longo devolve mais do topo. Não há distância certa, há a que combina com o seu prazo.


## Para ir além
1. Na Parte 1, compare a diferença por ano de cada ETF com a taxa de administração da Parte 3. Qual se afasta mais da própria taxa?
2. Na Parte 2, calcule a diferença do BITO ano a ano. Em que anos o custo da rolagem foi maior?
3. Na Parte 4, troque o Bitcoin pelo Ether. A cesta ganhou do Ether?
4. Na Parte 5, recalcule tudo a partir do topo da MSTR. Qual caiu mais, a ação ou a moeda?
5. Na Parte 6, troque o ganho por uma perda de 30%. Em qual caminho o prejuízo ainda serve para alguma coisa?
6. Na Parte 8, troque a janela de 30 para 7 dias. A alavancagem de 10x fica segura?
7. Na Parte 10, quanto custa a put com strike de R$ 350 mil? E por 6 meses?
8. Na Parte 11, apague a foto do livro e tire outra. O spread e a profundidade mudaram?
9. Na Parte 13, troque o fechamento pela mínima do dia. Os stops saem antes ou depois?

**Governança:** os resultados valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo.

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros.

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido.

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.